In [19]:
import pandas as pd
import numpy as np
import sqlalchemy as db
from src.get_all_data import get_all_data
import joblib

In [20]:
engine = db.create_engine('sqlite:///../data/raw/data.db')
df = get_all_data(engine)

Searching for events

[+] US Federal Funds Rate
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Statement
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Press Conference
Skipped — next release on 2026-04-29 (not yet passed)

[+] US FOMC Economic Projections
Skipped — next release on 2026-06-17 (not yet passed)

[+] US Core CPI m/m
Skipped — next release on 2026-04-10 (not yet passed)

[+] US CPI m/m
Skipped — next release on 2026-04-10 (not yet passed)

[+] US CPI y/y
Skipped — next release on 2026-04-10 (not yet passed)

[+] US PPI m/m
Skipped — next release on 2026-04-14 (not yet passed)

[+] US Core PCE Price Index m/m
Skipped — next release on 2026-04-09 (not yet passed)

[+] US Non-Farm Employment Change
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Unemployment Rate
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Average Hourly Earnings m/m
Skipped — next release on 2026-04-03 (not yet passed)

[+] US Advance GD

$DX=F: possibly delisted; no price data found  (1d 1927-04-14 -> 2026-03-20) (Yahoo error = "No data found, symbol may be delisted")


[DX=F] Empty response on attempt 1/3, retrying in 5s...


$DX=F: possibly delisted; no price data found  (1d 1927-04-14 -> 2026-03-20) (Yahoo error = "No data found, symbol may be delisted")


[DX=F] Empty response on attempt 2/3, retrying in 5s...


$DX=F: possibly delisted; no price data found  (1d 1927-04-14 -> 2026-03-20) (Yahoo error = "No data found, symbol may be delisted")


[DX=F] Empty response on attempt 3/3, retrying in 5s...
[DX=F] Failed after 3 attempts — skipping.
Fetching GC=F...
[GC=F] Already up to date, no new rows to insert.

The following tickers had issues: ['DX=F']
Processing database
All done


In [21]:
hmm = joblib.load('../models/hmm_model.pkl')
pca = joblib.load('../models/pca.pkl')
scaler = joblib.load('../models/scaler.pkl')

X_scaled = scaler.transform(df)
X_pca    = pca.transform(X_scaled)


n_states = hmm.n_components
state_probs = hmm.predict_proba(X_pca)
state_predict = hmm.predict(X_pca)


df['state'] = state_predict

for i in range(n_states):
    df[f'prob_state_{i}'] = state_probs[:, i]

In [22]:
df['spy_return'] = df['spy_close'].pct_change()
df['qqq_return'] = df['qqq_close'].pct_change()
df['^vix_return'] = df['^vix_close'].pct_change()
df['dx-y.nyb_return'] = df['dx-y.nyb_close'].pct_change()
df['gc=f_return'] = df['gc=f_close'].pct_change()

return_cols = {
    'spy_return':       'SPY',
    'qqq_return':       'QQQ',
    '^vix_return':      'VIX',
    'dx-y.nyb_return':  'DXY',
    'gc=f_return':      'Gold',
}

rows = []
for state in sorted(df['state'].unique()):
    mask = df['state'] == state
    row = {'state': state, 'count': mask.sum()}
    for col, name in return_cols.items():
        ret = df.loc[mask, col]
        row[f'{name}_mean']   = ret.mean()
        row[f'{name}_std']    = ret.std()
        row[f'{name}_sharpe'] = ret.mean() / ret.std()
    rows.append(row)

regime_profile = pd.DataFrame(rows).set_index('state')

#see all states
df_reset = regime_profile.reset_index()
df_long = df_reset.melt(id_vars='state', var_name='variable', value_name='value')

df_long[['asset', 'metric']] = df_long['variable'].str.split('_', n = 1 ,expand=True)

df_final = df_long.pivot_table(
    index=['state', 'asset'],
    columns='metric',
    values='value'
).reset_index()

df_final.columns.name = None
df_final = df_final[['state', 'asset', 'mean', 'std', 'sharpe']]

for state, group in df_final.groupby('state'):
    print(f"\nSTATE {state}")
    print(group.to_string(index= False))


STATE 0
 state asset      mean      std    sharpe
     0   DXY -0.000036 0.006280 -0.005788
     0  Gold  0.000073 0.018518  0.003922
     0   QQQ  0.004325 0.023161  0.186757
     0   SPY  0.003404 0.020704  0.164411
     0   VIX -0.003622 0.124353 -0.029128

STATE 1
 state asset      mean      std    sharpe
     1   DXY  0.000082 0.008726  0.009438
     1  Gold  0.002141 0.021334  0.100378
     1   QQQ -0.001291 0.027516 -0.046905
     1   SPY -0.001474 0.025043 -0.058857
     1   VIX  0.021022 0.153211  0.137208

STATE 2
 state asset     mean      std   sharpe
     2   DXY 0.000044 0.006921 0.006424
     2  Gold 0.000594 0.017316 0.034288
     2   QQQ 0.001494 0.016223 0.092065
     2   SPY 0.001126 0.013934 0.080799
     2   VIX 0.003950 0.093283 0.042346

STATE 3
 state asset     mean      std   sharpe
     3   DXY 0.000258 0.006202 0.041585
     3  Gold 0.000503 0.011989 0.041928
     3   QQQ 0.001824 0.014617 0.124793
     3   SPY 0.001331 0.011546 0.115313
     3   VIX 0.00313

In [23]:
macro_cols = {
    'US Federal Funds Rate':          'Fed Funds',
    'US Unemployment Rate':           'Unemployment',
    'US Non-Farm Employment Change':  'NFP',
    'US Housing Starts':              'Housing Starts',
    'US CPI m/m':                     'CPI m/m',
    'US Core CPI m/m':                'Core CPI',
    'US ISM Manufacturing PMI':       'ISM Mfg',
    'US Retail Sales m/m':            'Retail Sales',
    'BAMLH0A0HYM2':                   'HY Spread',
    'BAMLC0A0CM':                     'IG Spread',
}

rows = []
for state in sorted(df['state'].unique()):
    mask = df['state'] == state
    row = {'state': state, 'count': mask.sum()}
    for col, name in macro_cols.items():
        row[f'{name}'] = df.loc[mask, col].mean()
    rows.append(row)

macro_profile = pd.DataFrame(rows).set_index('state')

print(macro_profile.round(4).to_string())

       count  Fed Funds  Unemployment          NFP  Housing Starts  CPI m/m  Core CPI  ISM Mfg  Retail Sales  HY Spread  IG Spread
state                                                                                                                             
0        179     0.0045        0.0767 -395782.1229    1.398268e+06   0.0021    0.0018  54.9955        0.0098     4.9219     1.3865
1        428     0.0167        0.0517  145341.1215    1.261028e+06   0.0027    0.0032  51.0404        0.0004     6.5498     2.1138
2        763     0.0101        0.0758   78111.4024    8.376409e+05   0.0022    0.0015  53.6163        0.0035     5.6771     1.8022
3        601     0.0122        0.0446  191222.9617    1.193627e+06   0.0013    0.0017  54.1238        0.0026     4.5205     1.3247
4        300     0.0510        0.0390  211063.3333    1.380867e+06   0.0025    0.0029  47.9790        0.0027     3.4455     1.0267


In [24]:

asset_config = {
    'spy_return':      {'weight': 0.20, 'direction':  1},
    'qqq_return':      {'weight': 0.15, 'direction':  1},
    '^vix_return':     {'weight': 0.15, 'direction': -1}, #VIX hihg = bad
    'dx-y.nyb_return': {'weight': 0.05, 'direction':  1},
    'gc=f_return':     {'weight': 0.05, 'direction':  1},
}

macro_config = {
    # Monetary
    'US Federal Funds Rate':         {'weight': 0.05, 'direction': -1},  # high rates = bad
    # Employment
    'US Unemployment Rate':          {'weight': 0.05, 'direction': -1},  # high unemployment = bad
    'US Non-Farm Employment Change': {'weight': 0.05, 'direction':  1},  # more employment = good
    # Activity
    'US ISM Manufacturing PMI':      {'weight': 0.05, 'direction':  1},  # >50 = expansion
    'US Housing Starts':             {'weight': 0.03, 'direction':  1},  # more builds = good
    'US Retail Sales m/m':           {'weight': 0.03, 'direction':  1},  # strong expendings = good
    # Inflation
    'US CPI m/m':                    {'weight': 0.02, 'direction': -1},  # high inflation = bad
    'US Core CPI m/m':               {'weight': 0.02, 'direction': -1},
    # Credits
    'BAMLH0A0HYM2':                  {'weight': 0.05, 'direction': -1},  # HY spread high = bad
    'BAMLC0A0CM':                    {'weight': 0.05, 'direction': -1},  # IG spread high = bad
}

# wheights must sum 1
total_weight = (
    sum(c['weight'] for c in asset_config.values()) +
    sum(c['weight'] for c in macro_config.values())
)
print(f"Total weight: {total_weight:.2f}")


#macro stats
macro_global_stats = {}
for col in macro_config:
    macro_global_stats[col] = {
        'mean': df[col].mean(),
        'std':  df[col].std(),
    }


#composite score by state
rows = []
for state in sorted(df['state'].unique()):
    mask = df['state'] == state
    row  = {'state': state, 'n_days': mask.sum()}
    composite = 0.0

    # Market: Sharpe adjust by direction
    for col, cfg in asset_config.items():
        ret        = df.loc[mask, col].dropna()
        sharpe     = ret.mean() / ret.std() if ret.std() > 0 else 0
        sharpe_adj = sharpe * cfg['direction']
        row[f'asset_{col}'] = sharpe_adj
        composite += sharpe_adj * cfg['weight']

    # Macro: z-score adjust by direction
    for col, cfg in macro_config.items():
        state_mean = df.loc[mask, col].mean()
        g_mean     = macro_global_stats[col]['mean']
        g_std      = macro_global_stats[col]['std']
        zscore     = (state_mean - g_mean) / g_std if g_std > 0 else 0
        zscore_adj = zscore * cfg['direction']
        row[f'macro_{col}'] = zscore_adj
        composite += zscore_adj * cfg['weight']

    row['composite_score'] = composite
    rows.append(row)

signal_df = pd.DataFrame(rows).set_index('state')

max_abs = signal_df['composite_score'].abs().max()
signal_df['signal_size'] = signal_df['composite_score'] / max_abs


#results
print(signal_df[['n_days', 'composite_score', 'signal_size']].round(4))


signal_map = signal_df['signal_size'].to_dict()
df['primary_signal'] = df['state'].map(signal_map)

Total weight: 1.00
       n_days  composite_score  signal_size
state                                      
0         179           0.1154       0.7980
1         428          -0.0949      -0.6567
2         763          -0.0270      -0.1868
3         601           0.1446       1.0000
4         300           0.0483       0.3342


In [27]:
df['spy_return_next'] = df['spy_return'].shift(-1)
#vix is an indicator
asset_return_config = {
    'spy_return':      {'weight': 0.40, 'direction':  1},
    'qqq_return':      {'weight': 0.30, 'direction':  1},
    'dx-y.nyb_return': {'weight': 0.15, 'direction':  1},
    'gc=f_return':     {'weight': 0.15, 'direction':  1},
}

df['composite_return_next'] = sum(
    df[col].shift(-1) * cfg['direction'] * cfg['weight']
    for col, cfg in asset_return_config.items()
)

#meta-label
df['meta_label'] = (
    np.sign(df['primary_signal']) == np.sign(df['composite_return_next'])
).astype(int)

#Verify
check = df.groupby('state').agg(
    n             = ('meta_label',           'count'),
    signal_size   = ('primary_signal',       'first'),
    pct_right     = ('meta_label',           'mean'),
    spy_ret       = ('spy_return_next',      'mean'),
    comp_ret      = ('composite_return_next','mean'),
).round(4)

print(check)

         n  signal_size  pct_right  spy_ret  comp_ret
state                                                
0      179       0.7980     0.6201   0.0038    0.0029
1      428      -0.6567     0.4533  -0.0005    0.0000
2      763      -0.1868     0.4194   0.0010    0.0009
3      601       1.0000     0.5857   0.0008    0.0008
4      300       0.3342     0.5800   0.0016    0.0016
